In [6]:
#pip install duckdb pandas pyarrow jupyter

In [7]:
import duckdb
import pandas as pd

con = duckdb.connect()

In [8]:
file_path = "fhvhv_tripdata_2024-01.parquet"

con.sql(f"""
SELECT *
FROM read_parquet('{file_path}')
LIMIT 10
""").df()

,hvfhs_license_num,dispatching_base_num,originating_base_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,...,sales_tax,congestion_surcharge,airport_fee,tips,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag
0,HV0003,B03404,B03404,2024-01-01 00:21:47,2024-01-01 00:25:06,2024-01-01 00:28:08,2024-01-01 01:05:39,161,158,2.83,...,4.05,2.75,0.0,0.0,40.18,N,N,N,N,N
1,HV0003,B03404,B03404,2024-01-01 00:10:56,2024-01-01 00:11:08,2024-01-01 00:12:53,2024-01-01 00:20:05,137,79,1.57,...,0.89,2.75,0.0,0.0,6.12,N,N,N,N,N
2,HV0003,B03404,B03404,2024-01-01 00:20:04,2024-01-01 00:21:51,2024-01-01 00:23:05,2024-01-01 00:35:16,79,186,1.98,...,1.60,2.75,0.0,0.0,9.47,N,N,N,N,N
3,HV0003,B03404,B03404,2024-01-01 00:35:46,2024-01-01 00:39:59,2024-01-01 00:41:04,2024-01-01 00:56:34,234,148,1.99,...,1.52,2.75,0.0,0.0,11.35,N,N,N,N,N
4,HV0003,B03404,B03404,2024-01-01 00:48:19,2024-01-01 00:56:23,2024-01-01 00:57:21,2024-01-01 01:10:02,148,97,2.65,...,3.43,2.75,0.0,0.0,28.63,N,N,N,N,N
5,HV0003,B03404,B03404,2024-01-01 00:03:47,2024-01-01 00:05:53,2024-01-01 00:06:15,2024-01-01 00:27:53,255,95,7.02,...,2.85,0.00,0.0,0.0,24.35,N,N,N,N,Y
6,HV0003,B03404,B03404,2024-01-01 00:22:51,2024-01-01 00:29:17,2024-01-01 00:29:47,2024-01-01 00:50:08,95,212,11.33,...,4.68,0.00,0.0,0.0,30.98,N,N,N,N,Y
7,HV0003,B03404,B03404,2024-01-01 00:45:34,2024-01-01 00:57:29,2024-01-01 00:57:50,2024-01-01 01:11:27,213,47,3.43,...,2.06,0.00,0.0,0.0,20.73,N,N,N,N,Y
8,HV0003,B03404,B03404,2024-01-01 00:11:51,2024-01-01 00:15:46,2024-01-01 00:16:00,2024-01-01 00:28:13,209,114,1.54,...,1.37,2.75,0.0,0.0,10.40,N,N,N,N,Y
9,HV0003,B03404,B03404,2024-01-01 00:26:48,2024-01-01 00:33:02,2024-01-01 00:33:15,2024-01-01 00:46:39,113,209,1.72,...,1.21,2.75,0.0,0.0,11.38,N,N,N,N,Y


### Cek Schema

In [9]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{file_path}')
""").df()

,column_name,column_type,null,key,default,extra
0,hvfhs_license_num,VARCHAR,YES,None,None,None
1,dispatching_base_num,VARCHAR,YES,None,None,None
2,originating_base_num,VARCHAR,YES,None,None,None
3,request_datetime,TIMESTAMP,YES,None,None,None
4,on_scene_datetime,TIMESTAMP,YES,None,None,None
5,pickup_datetime,TIMESTAMP,YES,None,None,None
6,dropoff_datetime,TIMESTAMP,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,trip_miles,DOUBLE,YES,None,None,None


### cek provider

In [10]:
con.sql(f"""
SELECT
    hvfhs_license_num,
    COUNT(*) AS total_trips
FROM read_parquet('{file_path}')
GROUP BY hvfhs_license_num
ORDER BY total_trips DESC
""").df()

,hvfhs_license_num,total_trips
0,HV0003,14432755
1,HV0005,5231175


### Cek Data quality awal

In [11]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE WHEN request_datetime IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_request_datetime,

    SUM(
        CASE WHEN PULocationID IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_pickup_location,

    SUM(
        CASE
            WHEN PULocationID < 1
              OR PULocationID > 265
            THEN 1
            ELSE 0
        END
    ) AS invalid_pickup_location

FROM read_parquet('{file_path}')
""").df()

,total_rows,missing_request_datetime,missing_pickup_location,invalid_pickup_location
0,19663930,0.0,0.0,0.0


### Validasi periode datanya

In [12]:
date_range = con.sql(f"""
SELECT
    MIN(request_datetime) AS min_request_datetime,
    MAX(request_datetime) AS max_request_datetime,
    MIN(pickup_datetime) AS min_pickup_datetime,
    MAX(pickup_datetime) AS max_pickup_datetime,
    MIN(dropoff_datetime) AS min_dropoff_datetime,
    MAX(dropoff_datetime) AS max_dropoff_datetime
FROM read_parquet('{file_path}')
""").df()

date_range

,min_request_datetime,max_request_datetime,min_pickup_datetime,max_pickup_datetime,min_dropoff_datetime,max_dropoff_datetime
0,2023-12-31 23:01:55,2024-02-01 00:07:07,2024-01-01,2024-01-31 23:59:59,2024-01-01 00:02:35,2024-02-01 02:10:41


In [13]:
datetime_quality = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN pickup_datetime < request_datetime
            THEN 1 ELSE 0
        END
    ) AS pickup_before_request,

    SUM(
        CASE
            WHEN dropoff_datetime < pickup_datetime
            THEN 1 ELSE 0
        END
    ) AS dropoff_before_pickup,

    SUM(
        CASE
            WHEN pickup_datetime = dropoff_datetime
            THEN 1 ELSE 0
        END
    ) AS zero_trip_duration

FROM read_parquet('{file_path}')
""").df()

datetime_quality

,total_rows,pickup_before_request,dropoff_before_pickup,zero_trip_duration
0,19663930,188251.0,0.0,2.0


In [15]:
month_boundary_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN request_datetime < TIMESTAMP '2024-01-01 00:00:00'
            THEN 1 ELSE 0
        END
    ) AS request_before_january,

    SUM(
        CASE
            WHEN request_datetime >= TIMESTAMP '2024-02-01 00:00:00'
            THEN 1 ELSE 0
        END
    ) AS request_after_january,

    SUM(
        CASE
            WHEN dropoff_datetime >= TIMESTAMP '2024-02-01 00:00:00'
            THEN 1 ELSE 0
        END
    ) AS dropoff_after_january

FROM read_parquet('{file_path}')
""").df()

month_boundary_check

,total_rows,request_before_january,request_after_january,dropoff_after_january
0,19663930,858.0,3.0,6584.0



### Hasil Data Quality Temporal

* Total data Januari 2024: **19.663.930 trip**.
* Tidak ada missing value pada `request_datetime` dan `PULocationID`.
* Tidak ada invalid `PULocationID`.
* Ditemukan **188.251 trip (±0,96%)** dengan `pickup_datetime` lebih awal dari `request_datetime`.
* Tidak ditemukan `dropoff_datetime` lebih awal dari `pickup_datetime`.
* Hanya terdapat **2 trip** dengan durasi waktu nol.
* Perbedaan tanggal di batas bulan masih dianggap wajar karena request, pickup, dan drop-off dapat terjadi pada hari/bulan yang berbeda.

### Keputusan

Temporal anomaly tersebut **tidak langsung dihapus** karena target forecasting menggunakan `request_datetime` dan `PULocationID`. Record dengan pickup sebelum request hanya akan dikecualikan jika nanti digunakan untuk analisis waiting time.



### Periksa missing value secara lebih lengkap

In [16]:
summary = con.sql(f"""
SUMMARIZE
SELECT *
FROM read_parquet('{file_path}')
""").df()

summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,hvfhs_license_num,VARCHAR,HV0003,HV0005,1,None,None,None,None,None,19663930,0.00
1,dispatching_base_num,VARCHAR,B03404,B03406,2,None,None,None,None,None,19663930,0.00
2,originating_base_num,VARCHAR,B00887,B03406,6,None,None,None,None,None,19663930,26.54
3,request_datetime,TIMESTAMP,2023-12-31 23:01:55,2024-02-01 00:07:07,2173382,2024-01-16 22:16:37.965606,None,2024-01-09 17:48:42.12378,2024-01-17 07:58:44.151776,2024-01-24 11:09:23.686326,19663930,0.00
4,on_scene_datetime,TIMESTAMP,2023-12-31 22:55:22,2024-01-31 23:59:58,2170337,2024-01-17 02:22:41.626252,None,2024-01-09 20:50:23.389102,2024-01-17 11:57:37.52925,2024-01-24 17:16:12.842327,19663930,26.54
5,pickup_datetime,TIMESTAMP,2024-01-01 00:00:00,2024-01-31 23:59:59,2207450,2024-01-16 22:20:45.530687,None,2024-01-09 13:49:45.683506,2024-01-17 09:23:13.011584,2024-01-24 15:32:54.397231,19663930,0.00
6,dropoff_datetime,TIMESTAMP,2024-01-01 00:02:35,2024-02-01 02:10:41,2326816,2024-01-16 22:39:15.999914,None,2024-01-09 16:24:41.802862,2024-01-17 08:17:13.074241,2024-01-24 12:15:28.951378,19663930,0.00
7,PULocationID,INTEGER,1,265,298,139.13607310440995,74.90266882758335,76,140,210,19663930,0.00
8,DOLocationID,INTEGER,1,265,298,142.4193262994732,77.77902811192249,76,142,218,19663930,0.00
9,trip_miles,DOUBLE,0.0,417.62,48348,4.8385865080886825,5.698662365016966,1.4947736176979083,2.831293655677645,5.98788092441353,19663930,0.00



### Data Quality Summary dan Interpretasi

Pemeriksaan awal dilakukan terhadap 24 variabel pada data HVFHV Januari 2024 yang berjumlah **19.663.930 perjalanan**. Pemeriksaan mencakup kelengkapan data, rentang nilai, distribusi awal, konsistensi waktu, lokasi, variabel finansial, serta kategori layanan. Secara umum, dataset memiliki tingkat kelengkapan yang baik karena sebagian besar variabel tidak memiliki missing value. Namun, beberapa variabel memerlukan perhatian lebih lanjut sebelum digunakan dalam analisis bisnis.

### 1. Kelengkapan Data

Sebanyak **22 dari 24 variabel memiliki missing value sebesar 0%**. Hal ini menunjukkan bahwa sebagian besar informasi perjalanan tercatat secara lengkap, termasuk variabel yang sangat penting untuk forecasting yaitu `request_datetime` dan `PULocationID`.

`request_datetime` menunjukkan waktu ketika customer melakukan permintaan perjalanan, sedangkan `PULocationID` menunjukkan area tempat permintaan tersebut berasal. Kedua variabel ini nantinya digunakan untuk membentuk target:

> **jumlah ride request pada setiap pickup zone per jam.**

Karena kedua variabel tersebut tidak memiliki missing value, tidak diperlukan imputasi maupun penghapusan data pada tahap pembentukan target demand.

Missing value sebesar **26,54%** ditemukan pada `originating_base_num` dan `on_scene_datetime`.

`originating_base_num` menunjukkan base yang pertama kali menerima atau menginisiasi request perjalanan. Variabel ini lebih berkaitan dengan proses dispatch atau administrasi provider dan **bukan variabel utama untuk mengukur jumlah demand customer**. Oleh karena itu, missing value pada variabel ini tidak menghambat pembentukan target forecasting.

`on_scene_datetime` menunjukkan waktu ketika kendaraan atau driver tiba di lokasi pickup. Variabel tersebut lebih relevan apabila project melakukan analisis khusus seperti waktu respons driver atau proses sebelum passenger dijemput. Untuk forecasting demand, waktu utama yang digunakan adalah `request_datetime`, sehingga missing value pada `on_scene_datetime` juga tidak secara langsung mempengaruhi target model.

Karena kedua variabel tersebut bukan variabel utama forecasting dan masih dapat memiliki kegunaan untuk analisis operasional tertentu, **kolom maupun record-nya belum dihapus pada tahap ini**. Penyebab missing value juga belum diasumsikan sebelum dilakukan pemeriksaan apabila variabel tersebut nantinya benar-benar dibutuhkan.

---

### 2. Konsistensi Variabel Lokasi

`PULocationID` dan `DOLocationID` memiliki rentang nilai **1 hingga 265**.

`PULocationID` merupakan ID zona tempat passenger dijemput, sedangkan `DOLocationID` merupakan ID zona tempat passenger diturunkan.

Pada pemeriksaan sebelumnya juga tidak ditemukan missing maupun nilai lokasi di luar rentang tersebut. Dengan demikian, variabel lokasi yang akan menjadi komponen utama analisis spatial demand dapat digunakan tanpa treatment khusus pada tahap ini.

Nilai `approx_unique` dari fungsi `SUMMARIZE` tidak digunakan sebagai jumlah kategori sebenarnya karena sifatnya hanya **estimasi**. Hal ini menjelaskan mengapa hasilnya dapat menunjukkan angka yang tampak tidak konsisten dengan rentang ID. Jika jumlah zona unik diperlukan, pemeriksaan akan menggunakan `COUNT(DISTINCT ...)`.

---

### 3. `trip_miles` — Jarak Perjalanan

`trip_miles` menunjukkan **jarak perjalanan passenger dalam satuan mile**.

Hasil summary menunjukkan:

* minimum = **0 mile**
* median = sekitar **2,83 miles**
* rata-rata = sekitar **4,84 miles**
* kuartil 75% = sekitar **5,99 miles**
* maksimum = **417,62 miles**

Median 2,83 miles menunjukkan bahwa setengah perjalanan memiliki jarak sekitar 2,83 miles atau kurang. Bahkan 75% perjalanan masih berada di bawah sekitar 6 miles.

Karena itu, nilai maksimum **417,62 miles** sangat jauh dibandingkan pola sebagian besar perjalanan.

Hal ini belum otomatis berarti data salah. Namun, nilai tersebut dapat dikategorikan sebagai **extreme value yang perlu dievaluasi**, karena jaraknya jauh lebih besar dibandingkan perjalanan ride-hailing tipikal dalam dataset.

Sementara itu, nilai minimum 0 mile menunjukkan bahwa terdapat perjalanan yang tercatat tanpa jarak perjalanan. Kondisi tersebut juga perlu dihitung jumlahnya. Zero distance bisa berasal dari perjalanan sangat pendek, pembatalan, kesalahan pencatatan, atau kondisi operasional lainnya.

Oleh karena itu, tahap berikutnya akan memeriksa **berapa banyak perjalanan dengan `trip_miles = 0` serta bagaimana distribusi extreme trip distance**, sebelum memutuskan apakah diperlukan filtering.

---

### 4. `trip_time` — Durasi Perjalanan

`trip_time` menunjukkan **lama perjalanan passenger dalam satuan detik**.

Hasil summary menunjukkan:

* minimum = **0 detik**
* median = **913 detik**
* rata-rata = sekitar **1.110 detik**
* kuartil 75% = **1.430 detik**
* maksimum = **52.060 detik**

Median 913 detik setara dengan sekitar **15 menit**, sedangkan rata-ratanya sekitar **18,5 menit**. Dengan demikian, sebagian besar perjalanan memiliki durasi yang relatif masuk akal untuk perjalanan perkotaan.

Namun, nilai maksimum 52.060 detik setara dengan sekitar **14,5 jam**, yang sangat jauh dari pola mayoritas perjalanan.

Nilai tersebut tidak langsung dinyatakan salah, tetapi termasuk **extreme duration** yang perlu diperiksa lebih lanjut. Perjalanan selama lebih dari 14 jam secara teknis mungkin saja tercatat, tetapi sangat tidak tipikal dibandingkan distribusi dataset.

Selain itu, terdapat nilai minimum 0 detik. Perjalanan dengan durasi nol tidak menggambarkan perjalanan normal sehingga jumlah kasus tersebut perlu dihitung sebelum menentukan treatment.

---

### 5. `base_passenger_fare` — Tarif Dasar Passenger

`base_passenger_fare` menunjukkan **tarif dasar yang dikenakan kepada passenger**, sebelum beberapa komponen seperti toll, tips, tax, dan surcharge diperhitungkan.

Hasil summary menunjukkan:

* minimum = **-43,09**
* median = sekitar **18,03**
* rata-rata = sekitar **23,96**
* maksimum = **1.911,16**

Mayoritas fare berada pada tingkat yang jauh lebih rendah dibandingkan nilai maksimum. Namun, terdapat dua hal yang perlu diperhatikan.

Pertama, terdapat **fare negatif sebesar -43,09**. Secara sederhana, customer tidak biasanya membayar tarif perjalanan dalam nilai negatif. Akan tetapi, nilai negatif belum boleh langsung dianggap error karena dapat berkaitan dengan adjustment, correction, refund, atau mekanisme pencatatan transaksi tertentu.

Kedua, terdapat fare maksimum **1.911,16**, yang sangat jauh dibandingkan median sekitar 18. Nilai tersebut menjadi kandidat extreme value yang perlu dievaluasi sebelum digunakan dalam KPI seperti average fare atau total passenger fare.

Hal ini penting karena beberapa nilai ekstrem dapat menarik nilai rata-rata ke atas dan membuat dashboard memberikan gambaran yang kurang representatif terhadap perjalanan normal.

---

### 6. `driver_pay` — Pembayaran kepada Driver

`driver_pay` menunjukkan jumlah pembayaran yang diterima driver untuk perjalanan yang dilakukan.

Hasil summary menunjukkan:

* minimum = **-67,76**
* median = sekitar **13,58**
* rata-rata = sekitar **18,27**
* maksimum = **1.218,17**

Mayoritas pembayaran driver berada jauh di bawah nilai maksimum tersebut.

Nilai negatif **-67,76** perlu diperiksa karena secara bisnis pembayaran kepada driver umumnya diharapkan bernilai positif. Namun, seperti passenger fare, nilai negatif belum langsung dihapus karena bisa berasal dari correction atau adjustment pada sistem transaksi.

Nilai maksimum **1.218,17** juga merupakan extreme value dibandingkan median sekitar 13,58. Apabila variabel ini akan digunakan untuk menghitung average driver pay pada dashboard, nilai ekstrem tersebut perlu dievaluasi agar tidak menghasilkan KPI yang misleading.

---

### 7. `tolls`, `congestion_surcharge`, `airport_fee`, dan `tips`

Beberapa komponen biaya memiliki median atau kuartil yang bernilai 0.

Contohnya:

* median `tolls` = **0**
* median `congestion_surcharge` = **0**
* median `airport_fee` = **0**
* median `tips` = **0**

Nilai nol pada variabel ini **tidak otomatis merupakan masalah data**.

Misalnya, tidak semua perjalanan melewati jalan tol, sehingga `tolls = 0` merupakan kondisi bisnis yang normal.

Demikian pula tidak semua perjalanan berasal dari atau menuju airport, sehingga `airport_fee = 0` juga normal.

Tips bersifat tidak wajib sehingga `tips = 0` juga dapat merepresentasikan perjalanan yang benar-benar tidak memperoleh tip.

Dengan demikian, zero value pada variabel-variabel tersebut tidak dapat diperlakukan sama seperti `trip_time = 0` atau `trip_miles = 0`.

Ini penting dalam cleaning:

> **angka nol harus diinterpretasikan berdasarkan makna bisnis setiap variabel, bukan langsung dianggap missing atau anomaly.**

---

### 8. Variabel Shared Ride dan Accessibility

Variabel:

* `shared_request_flag`
* `shared_match_flag`
* `access_a_ride_flag`
* `wav_request_flag`
* `wav_match_flag`

memiliki kategori:

`Y` dan `N`

serta tidak mempunyai missing value.

Hal ini menunjukkan bahwa secara struktur kategori tidak ditemukan value yang tidak konsisten seperti kategori selain `Y/N`, blank, ataupun missing.

`shared_request_flag` menunjukkan apakah customer meminta shared ride, sedangkan `shared_match_flag` menunjukkan apakah request tersebut akhirnya benar-benar mendapatkan shared ride match.

Sementara itu, `wav_request_flag` dan `wav_match_flag` berkaitan dengan permintaan dan pemenuhan **Wheelchair Accessible Vehicle**.

Variabel ini tidak menjadi feature utama forecasting, tetapi dapat digunakan sebagai tambahan **operational/service analysis** pada dashboard jika hasil analisisnya memberikan business insight yang relevan.

---


### Numeric Quality Assessment


`zero_trip_miles`
Artinya perjalanan tercatat memiliki jarak **0 mile**. Ini perlu diperiksa karena perjalanan normal biasanya memiliki jarak lebih dari nol.

`zero_trip_time`
Artinya perjalanan tercatat berlangsung **0 detik**. Ini lebih mencurigakan karena secara bisnis perjalanan yang benar-benar terjadi seharusnya membutuhkan waktu.

`negative_fare`
Artinya `base_passenger_fare` bernilai negatif. Secara transaksi biasa hal ini tidak normal, tetapi bisa saja berhubungan dengan adjustment atau correction, jadi belum langsung dihapus.

`zero_fare`
Artinya tarif dasar perjalanan bernilai 0. Kita perlu mengetahui seberapa banyak kasusnya sebelum menentukan apakah itu anomaly.

`negative_driver_pay`
Artinya pembayaran kepada driver bernilai negatif. Ini juga perlu diperiksa sebelum digunakan dalam KPI finansial.

`zero_driver_pay`
Artinya driver tercatat menerima pembayaran 0 untuk trip tersebut.



In [18]:
numeric_quality = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    -- Trip Distance
    SUM(CASE WHEN trip_miles = 0 THEN 1 ELSE 0 END)
        AS zero_trip_miles,

    -- Trip Duration
    SUM(CASE WHEN trip_time = 0 THEN 1 ELSE 0 END)
        AS zero_trip_time,

    -- Passenger Fare
    SUM(CASE WHEN base_passenger_fare < 0 THEN 1 ELSE 0 END)
        AS negative_fare,

    SUM(CASE WHEN base_passenger_fare = 0 THEN 1 ELSE 0 END)
        AS zero_fare,

    -- Driver Pay
    SUM(CASE WHEN driver_pay < 0 THEN 1 ELSE 0 END)
        AS negative_driver_pay,

    SUM(CASE WHEN driver_pay = 0 THEN 1 ELSE 0 END)
        AS zero_driver_pay

FROM read_parquet('{file_path}')
""").df()

numeric_quality

,total_rows,zero_trip_miles,zero_trip_time,negative_fare,zero_fare,negative_driver_pay,zero_driver_pay
0,19663930,3286.0,2.0,191.0,7143.0,137.0,4496.0



### Interpretasi Hasil Numeric Quality

| Temuan                    | Jumlah | Proporsi |
| ------------------------- | -----: | -------: |
| `trip_miles = 0`          |  3.286 |  0,0167% |
| `trip_time = 0`           |      2 | 0,00001% |
| `base_passenger_fare < 0` |    191 |  0,0010% |
| `base_passenger_fare = 0` |  7.143 |  0,0363% |
| `driver_pay < 0`          |    137 |  0,0007% |
| `driver_pay = 0`          |  4.496 |  0,0229% |

### 1. `trip_miles = 0` sebanyak 3.286 trip

`trip_miles` menunjukkan jarak perjalanan passenger.

Nilai 0 berarti terdapat perjalanan yang tercatat **tidak memiliki jarak tempuh**, padahal secara umum perjalanan ride-hailing seharusnya memiliki perpindahan dari lokasi pickup ke tujuan.

Namun jumlahnya hanya sekitar **0,017% dari keseluruhan data**, sehingga kasus ini sangat kecil.

Nilai 0 belum langsung dihapus karena bisa berkaitan dengan perjalanan sangat pendek, trip yang tidak selesai secara normal, atau mekanisme pencatatan tertentu.

**Keputusan sementara:** tandai sebagai anomali untuk analisis jarak. Record tidak perlu langsung dihapus dari dataset forecasting karena forecasting tidak menggunakan `trip_miles` sebagai target utama.

---

### 2. `trip_time = 0` hanya 2 trip

`trip_time` menunjukkan durasi perjalanan dalam detik.

Nilai 0 berarti perjalanan tercatat memiliki durasi **0 detik**, yang secara bisnis tidak merepresentasikan perjalanan normal.

Namun hanya terdapat **2 record dari hampir 20 juta perjalanan**.

**Keputusan:** record ini dapat dikecualikan ketika melakukan analisis durasi perjalanan. Dampaknya terhadap keseluruhan dataset praktis tidak material.

---

### 3. Negative passenger fare sebanyak 191 trip

`base_passenger_fare` menunjukkan tarif dasar yang dikenakan kepada passenger.

Ditemukan **191 trip dengan fare negatif** atau hanya sekitar **0,001% data**.

Secara transaksi perjalanan biasa, tarif negatif tidak menggambarkan pembayaran normal dari customer. Akan tetapi, kita belum dapat memastikan bahwa nilai tersebut merupakan kesalahan karena bisa saja berkaitan dengan refund, adjustment, correction, atau mekanisme pencatatan transaksi.

**Keputusan sementara:** jangan langsung dihapus. Namun record dengan fare negatif sebaiknya tidak langsung digunakan dalam KPI seperti average passenger fare sebelum treatment final ditentukan.

---

### 4. Zero passenger fare sebanyak 7.143 trip

Sebanyak **7.143 trip** memiliki `base_passenger_fare = 0`, atau sekitar **0,036% data**.

Artinya terdapat sejumlah kecil trip yang tercatat tanpa tarif dasar passenger.

Jumlahnya masih sangat kecil dibanding total data. Kondisi ini bisa berasal dari promotional trip, adjustment, cancellation, atau kondisi transaksi lainnya, tetapi dataset yang kita miliki belum cukup untuk langsung menentukan penyebabnya.

**Keputusan sementara:** dipertahankan dahulu, tetapi perlu diperhatikan ketika menghitung financial KPI.

---

### 5. Negative driver pay sebanyak 137 trip

`driver_pay` menunjukkan jumlah pembayaran kepada driver.

Ditemukan **137 trip dengan driver pay negatif**, hanya sekitar **0,0007% data**.

Secara operasional, pembayaran negatif bukan kondisi perjalanan normal. Namun sama seperti fare negatif, kemungkinan terdapat correction atau adjustment yang tidak dapat kita identifikasi hanya dari kolom yang tersedia.

**Keputusan sementara:** tidak langsung dihapus, tetapi tidak digunakan secara sembarangan dalam KPI average driver pay.

---

### 6. Zero driver pay sebanyak 4.496 trip

Sebanyak **4.496 trip** atau sekitar **0,023% data** memiliki `driver_pay = 0`.

Artinya sistem mencatat tidak adanya pembayaran driver pada trip tersebut.

Karena jumlahnya sangat kecil, kondisi ini tidak menunjukkan masalah kualitas data secara keseluruhan. Tetapi record tersebut perlu diperhatikan pada analisis finansial.

---

### Kesimpulan

> Pemeriksaan numeric data quality menunjukkan adanya beberapa zero dan negative values pada variabel perjalanan dan finansial. Dari total 19.663.930 perjalanan, terdapat 3.286 trip dengan jarak 0 mile dan 2 trip dengan durasi 0 detik. Pada variabel finansial ditemukan 191 perjalanan dengan passenger fare negatif, 7.143 perjalanan dengan passenger fare bernilai nol, 137 perjalanan dengan driver pay negatif, dan 4.496 perjalanan dengan driver pay bernilai nol.
>
> Meskipun secara bisnis beberapa nilai tersebut tidak merepresentasikan perjalanan atau transaksi normal, seluruh kasus memiliki proporsi yang sangat kecil, masing-masing kurang dari 0,04% dari keseluruhan data. Oleh karena itu, nilai tersebut tidak langsung dihapus dari keseluruhan dataset. Treatment akan disesuaikan dengan tujuan analisis. Sebagai contoh, zero-distance trip dapat dikecualikan dari analisis jarak dan zero-duration trip dari analisis durasi, sedangkan nilai fare dan driver pay yang tidak normal perlu diperhatikan ketika membentuk financial KPI. Temuan tersebut tidak secara langsung mempengaruhi demand forecasting karena target forecasting dibentuk menggunakan `request_datetime` dan `PULocationID`.



### Extreme Value Check

In [19]:
extreme_value_check = con.sql(f"""
WITH stats AS (
    SELECT
        approx_quantile(trip_miles, 0.99) AS miles_p99,
        approx_quantile(trip_miles, 0.999) AS miles_p999,

        approx_quantile(trip_time, 0.99) AS time_p99,
        approx_quantile(trip_time, 0.999) AS time_p999,

        approx_quantile(base_passenger_fare, 0.99) AS fare_p99,
        approx_quantile(base_passenger_fare, 0.999) AS fare_p999,

        approx_quantile(driver_pay, 0.99) AS pay_p99,
        approx_quantile(driver_pay, 0.999) AS pay_p999

    FROM read_parquet('{file_path}')
)

SELECT
    -- Trip Miles
    s.miles_p99,
    s.miles_p999,
    MAX(d.trip_miles) AS miles_max,
    SUM(CASE WHEN d.trip_miles > s.miles_p999 THEN 1 ELSE 0 END)
        AS miles_above_p999,

    -- Trip Time
    s.time_p99,
    s.time_p999,
    MAX(d.trip_time) AS time_max,
    SUM(CASE WHEN d.trip_time > s.time_p999 THEN 1 ELSE 0 END)
        AS time_above_p999,

    -- Passenger Fare
    s.fare_p99,
    s.fare_p999,
    MAX(d.base_passenger_fare) AS fare_max,
    SUM(CASE WHEN d.base_passenger_fare > s.fare_p999 THEN 1 ELSE 0 END)
        AS fare_above_p999,

    -- Driver Pay
    s.pay_p99,
    s.pay_p999,
    MAX(d.driver_pay) AS pay_max,
    SUM(CASE WHEN d.driver_pay > s.pay_p999 THEN 1 ELSE 0 END)
        AS pay_above_p999

FROM read_parquet('{file_path}') d
CROSS JOIN stats s

GROUP BY
    s.miles_p99,
    s.miles_p999,
    s.time_p99,
    s.time_p999,
    s.fare_p99,
    s.fare_p999,
    s.pay_p99,
    s.pay_p999
""").df()

extreme_value_check

,miles_p99,miles_p999,miles_max,miles_above_p999,time_p99,time_p999,time_max,time_above_p999,fare_p99,fare_p999,fare_max,fare_above_p999,pay_p99,pay_p999,pay_max,pay_above_p999
0,26.456553,53.349661,417.62,16638.0,3810,5814,52060,17004.0,105.1412,197.835949,1911.16,20387.0,74.943477,141.078216,1218.17,16739.0




### Interpretasi Extreme Value

| Variabel              |         P99 |       P99.9 |     Maksimum | > P99.9 | Proporsi |
| --------------------- | ----------: | ----------: | -----------: | ------: | -------: |
| `trip_miles`          | 26,46 miles | 53,35 miles | 417,62 miles |  16.638 |   0,085% |
| `trip_time`           | 3.810 detik | 5.814 detik | 52.060 detik |  17.004 |   0,086% |
| `base_passenger_fare` |      105,14 |      197,84 |     1.911,16 |  20.387 |   0,104% |
| `driver_pay`          |       74,94 |      141,08 |     1.218,17 |  16.739 |   0,085% |

### `trip_miles`

Sebanyak 99% perjalanan memiliki jarak sekitar **26,46 miles atau kurang**, dan 99,9% berada di bawah sekitar **53,35 miles**. Namun maksimum mencapai **417,62 miles**.

Artinya, perjalanan 417 miles memang **sangat ekstrem dibandingkan mayoritas perjalanan**, tetapi perjalanan panjang belum otomatis merupakan data error. Hanya sekitar 0,085% data yang berada di atas P99.9.

**Keputusan sementara:** tidak dihapus otomatis. Extreme distance akan diperlakukan sebagai outlier valid sampai ada bukti bahwa nilainya merupakan kesalahan pencatatan.

### `trip_time`

Sebanyak 99% perjalanan berlangsung maksimal sekitar **63,5 menit**, sedangkan 99,9% maksimal sekitar **96,9 menit**. Nilai maksimum mencapai **52.060 detik atau sekitar 14,5 jam**.

Durasi 14,5 jam sangat tidak tipikal untuk perjalanan ride-hailing, tetapi hanya sekitar 0,086% record yang berada di atas P99.9.

**Keputusan sementara:** tidak langsung dihapus. Namun extreme duration perlu diperhatikan apabila digunakan untuk KPI seperti average trip duration.

### `base_passenger_fare`

Sebanyak 99% perjalanan memiliki base fare sekitar **105 atau kurang**, dan 99,9% berada di bawah sekitar **197,84**. Nilai maksimum mencapai **1.911,16**.

Ini menunjukkan distribusi fare memiliki ekor kanan yang panjang. Fare maksimum tersebut sangat ekstrem dibanding mayoritas transaksi, tetapi hanya sekitar 0,10% data berada di atas P99.9.

**Keputusan sementara:** tidak dilakukan trimming otomatis karena fare tinggi masih dapat berhubungan dengan perjalanan yang memang sangat panjang atau kondisi transaksi tertentu.

### `driver_pay`

Sebanyak 99% driver pay berada di bawah sekitar **74,94**, sedangkan 99,9% berada di bawah sekitar **141,08**. Maksimum mencapai **1.218,17**.

Nilai maksimum sangat jauh dari pola mayoritas data, tetapi hanya sekitar 0,085% yang berada di atas P99.9.

**Keputusan sementara:** dipertahankan sementara dan tidak otomatis dianggap error.

---

> Kesimpulan

> Pemeriksaan extreme values menunjukkan adanya right-tail observations pada variabel jarak, durasi, passenger fare, dan driver pay. Sebanyak 99,9% perjalanan memiliki jarak tidak lebih dari sekitar 53,35 miles, durasi tidak lebih dari 5.814 detik atau sekitar 97 menit, passenger fare tidak lebih dari 197,84, serta driver pay tidak lebih dari 141,08. Meskipun ditemukan nilai maksimum yang jauh lebih tinggi, seperti jarak 417,62 miles, durasi sekitar 14,5 jam, passenger fare 1.911,16, dan driver pay 1.218,17, proporsi observasi di atas persentil 99,9 hanya sekitar 0,08–0,10% dari keseluruhan data. Oleh karena itu, nilai ekstrem tidak langsung dihapus karena outlier secara statistik tidak selalu menunjukkan data error. Treatment akan disesuaikan dengan kebutuhan analisis agar tidak menghilangkan perjalanan yang mungkin valid.



In [21]:
duplicate_check = con.sql(f"""
WITH duplicate_groups AS (
    SELECT
        *,
        COUNT(*) AS duplicate_count
    FROM read_parquet('{file_path}')
    GROUP BY ALL
    HAVING COUNT(*) > 1
)

SELECT
    COUNT(*) AS duplicate_groups,
    COALESCE(SUM(duplicate_count), 0) AS rows_in_duplicate_groups,
    COALESCE(SUM(duplicate_count - 1), 0) AS duplicate_rows
FROM duplicate_groups
""").df()

duplicate_check

,duplicate_groups,rows_in_duplicate_groups,duplicate_rows
0,0,0.0,0.0


Pemeriksaan duplicate dilakukan dengan mengidentifikasi record yang memiliki nilai identik pada seluruh variabel. Hasil pemeriksaan menunjukkan tidak terdapat exact duplicate pada 19.663.930 record Januari 2024. Oleh karena itu, tidak diperlukan penghapusan data akibat duplikasi.

| Temuan                         |            Hasil Jan 2024 | Keputusan Final                                                                                         | Berlaku untuk           |
| ------------------------------ | ------------------------: | ------------------------------------------------------------------------------------------------------- | ----------------------- |
| Missing `request_datetime`     |                         0 | Record dengan nilai NULL nantinya dikeluarkan dari demand forecasting                                   | Forecasting             |
| Missing `PULocationID`         |                         0 | Record NULL nantinya dikeluarkan dari demand forecasting                                                | Forecasting             |
| Invalid `PULocationID`         |                         0 | Location ID di luar range valid dikeluarkan                                                             | Forecasting & Analytics |
| Missing `originating_base_num` |                    26,54% | **Tidak diimputasi**; dipertahankan sebagai NULL jika dibutuhkan                                        | Analytics               |
| Missing `on_scene_datetime`    |                    26,54% | **Tidak diimputasi**; tidak digunakan untuk analisis yang membutuhkan waktu kedatangan driver jika NULL | Analytics               |
| Pickup sebelum request         |          188.251 (±0,96%) | Record tetap dipertahankan, tetapi tidak digunakan untuk waiting-time analysis                          | Analytics               |
| Drop-off sebelum pickup        |                         0 | Jika ditemukan pada bulan lain, exclude dari duration analysis                                          | Analytics               |
| `trip_miles = 0`               |                     3.286 | Tetap di master data, exclude dari KPI normal trip distance                                             | Analytics               |
| `trip_time = 0`                |                         2 | Tetap di master data, exclude dari duration KPI                                                         | Analytics               |
| Passenger fare < 0             |                       191 | Tetap di master data, exclude dari normal fare KPI                                                      | Analytics               |
| Passenger fare = 0             |                     7.143 | Tetap disimpan, tetapi dipisahkan dari KPI normal paid-trip fare                                        | Analytics               |
| Driver pay < 0                 |                       137 | Tetap di master data, exclude dari normal driver-pay KPI                                                | Analytics               |
| Driver pay = 0                 |                     4.496 | Tetap disimpan, tetapi diperhatikan terpisah dalam financial analysis                                   | Analytics               |
| Extreme numeric values         | ±0,08–0,10% di atas P99.9 | **Tidak dihapus otomatis**                                                                              | Analytics               |
| Exact duplicate                |                         0 | Jika ditemukan pada bulan lain, hapus salinan tambahan                                                  | Semua                   |
